In [1]:
from structure import build_ase_bilayer, symbols_to_types

atoms, fine_sym, coarse_sym = build_ase_bilayer(out_dir="out")
pos = atoms.get_positions()
types = symbols_to_types(atoms.get_chemical_symbols(), fine_sym, coarse_sym)

print(f"atoms={len(atoms)}  fine={fine_sym}  coarse={coarse_sym}")
print(f"AA={int((types == 1).sum())}  CG={int((types == 2).sum())}")
print(f"saved: out/1_full.xyz")
atoms

atoms=1152  fine=Al  coarse=Cu
AA=1024  CG=128
saved: out/1_full.xyz


Atoms(symbols='Al1024Cu128', pbc=False)

In [2]:
import jax.numpy as jnp
from jax_md import space

from constants import R_CUT_AA, R_CUT_CG, NBR_CUT
from energy import build_hybrid_energy, species_cutoff_matrix
from simulation import one_jax_step
from structure import orthorhombic_box

print(f"R_CUT_AA={R_CUT_AA}  R_CUT_CG={R_CUT_CG}  NBR_CUT={NBR_CUT}")
print(f"r_cutoff matrix (on B*d):\n{species_cutoff_matrix()}")

box = orthorhombic_box(pos)
R = jnp.array(pos, dtype=jnp.float64)
types_j = jnp.array(types, dtype=jnp.int32)
# displacement, _ = space.periodic(box)
displacement, _ = space.free()

nbr_fn, energy_fn = build_hybrid_energy(displacement, box, types_j, topo=None)
neighbor = nbr_fn.allocate(R)
V = jnp.zeros_like(R)

print(f"neighbor list: shape={neighbor.idx.shape}  max_occupancy={neighbor.max_occupancy}")

E0 = float(energy_fn(R, neighbor))
print(f"E_homogeneous = {E0:.6f}")

R1, V1, neighbor1, E_before, E_after = one_jax_step(R, V, neighbor, nbr_fn, energy_fn, dt=1e-4)
print(f"E after 1 step = {float(E_after):.6f}")
print(f"max |dr| = {float(jnp.max(jnp.linalg.norm(R1 - R, axis=1))):.3e}")

R_CUT_AA=5.0  R_CUT_CG=10.0  NBR_CUT=10.0
r_cutoff matrix (on B*d):
[[0. 0. 0.]
 [0. 5. 0.]
 [0. 0. 5.]]
neighbor list: shape=(1152, 296)  max_occupancy=296
E_homogeneous = -3389.850861
E after 1 step = -3389.850878
max |dr| = 2.860e-08


In [3]:
neighbor = nbr_fn.allocate(R)
print(neighbor.idx.shape)          # (1152, 1151)
print(neighbor.max_occupancy)       # 1151 — ёмкость
# сколько реально используется в первой строке:
print((neighbor.idx[0] >= 0).sum())  # или смотреть ненулевые индексы

(1152, 296)
296
296


In [4]:
atoms_spheres = neighbor.reference_position[neighbor.idx].shape
atoms_coords = neighbor.reference_position
atoms_types = types_j[neighbor.idx]

In [5]:
import numpy as np
from constants import TYPE_AA, TYPE_CG, R_CUT_AA, R_CUT_CG, NBR_CUT

n = len(types)
idx = np.asarray(neighbor.idx)
types_np = np.asarray(types)
R_np = np.asarray(R)

# pair cutoff matrix (physical Å): nbr list built with NBR_CUT = max
rc = np.zeros((3, 3))
rc[1, 1] = R_CUT_AA
rc[2, 2] = R_CUT_CG
rc[1, 2] = rc[2, 1] = max(R_CUT_AA, R_CUT_CG)

own = np.arange(n)[:, None]
valid = (idx >= 0) & (idx < n) & (idx != own)
safe_idx = np.clip(idx, 0, n - 1)

type_center = types_np[:, None]
type_neighbor = types_np[safe_idx]
pair_rc = rc[type_center, type_neighbor]
dr = np.linalg.norm(R_np[safe_idx] - R_np[:, None, :], axis=-1)
in_cutoff = valid & (dr < pair_rc)

cross_mask = (
    (in_cutoff & (type_neighbor == TYPE_AA)).any(axis=1)
    & (in_cutoff & (type_neighbor == TYPE_CG)).any(axis=1)
)
center_idx_cross = np.where(cross_mask)[0]

print(f"NBR_CUT={NBR_CUT}  rc[1,1]={R_CUT_AA}  rc[2,2]={R_CUT_CG}")
print(f"centers with AA+CG (after pair cutoff): {cross_mask.sum()} / {n}")
if len(center_idx_cross):
    i0 = center_idx_cross[0]
    m = in_cutoff[i0]
    print(
        f"example center {i0} (type={types_np[i0]}): "
        f"AA={((type_neighbor[i0]==TYPE_AA)&m).sum()} "
        f"CG={((type_neighbor[i0]==TYPE_CG)&m).sum()}"
    )

NBR_CUT=10.0  rc[1,1]=5.0  rc[2,2]=10.0
centers with AA+CG (after pair cutoff): 576 / 1152
example center 8 (type=1): AA=13 CG=2


In [6]:
pick = 15  # index into center_idx_cross
center_i = int(center_idx_cross[pick])

import numpy as np
from ase import Atoms
from ase.io import write
from constants import TYPE_AA, TYPE_CG

positions, symbols = [R_np[center_i]], ["Fe"]
m = in_cutoff[center_i]
row_idx = idx[center_i][m]
row_types = types_np[row_idx]

for j, t in zip(row_idx, row_types):
    positions.append(R_np[j])
    if t == TYPE_AA:
        symbols.append(fine_sym)
    elif t == TYPE_CG:
        symbols.append(coarse_sym)

out_path = f"out/nbr_sphere_{center_i}.xyz"
write(out_path, Atoms(symbols=symbols, positions=np.array(positions)))

n_aa = int((row_types == TYPE_AA).sum())
n_cg = int((row_types == TYPE_CG).sum())
print(f"pick={pick}  center_i={center_i}  center_type={types_np[center_i]}")
print(f"AA={n_aa}  CG={n_cg}  total={len(positions)}")
print(f"saved: {out_path}")

pick=15  center_i=31  center_type=1
AA=19  CG=8  total=28
saved: out/nbr_sphere_31.xyz


In [7]:
from ase import Atoms
from ase.io import write
from boundary import estimate_r0, pack_sphere, sphere_mask, unpack_sphere
from constants import TYPE_AA, TYPE_CG, NBR_CUT

center = R_np[center_i]
r0_fine = estimate_r0(pos[types == TYPE_AA])
r_cut_b = NBR_CUT  # same max cutoff as jax neighbor list

m_b = sphere_mask(pos, center, r_cut_b)
before_pos, before_typ = pos[m_b], types[m_b]
before_sym = [fine_sym if t == TYPE_AA else coarse_sym for t in before_typ]
before_path = f"out/nbr_boundary_{center_i}_before.xyz"
write(before_path, Atoms(symbols=before_sym, positions=before_pos))

if types_np[center_i] == TYPE_AA:
    after_pos, after_typ, _ = unpack_sphere(
        pos, types, center, r_cut_b, r0_fine, filter_at=center,
    )
    after_sym = [
        fine_sym if t == TYPE_AA else coarse_sym if t == TYPE_CG else "Ag"
        for t in after_typ
    ]
    mode = "unpack (situation A)"
    n_ph = int((after_typ == 0).sum())
elif types_np[center_i] == TYPE_CG:
    after_pos, after_typ, _ = pack_sphere(pos, types, center, r_cut_b)
    n_cg = int((after_typ == TYPE_CG).sum())
    after_sym = [coarse_sym] * n_cg + ["Au"] * (len(after_typ) - n_cg)
    mode = "pack (situation B)"
    n_ph = int(len(after_typ) - n_cg)
else:
    raise ValueError(f"unknown center type {types_np[center_i]}")

after_path = f"out/nbr_boundary_{center_i}_after.xyz"
write(after_path, Atoms(symbols=after_sym, positions=after_pos))

print(f"center_i={center_i}  mode={mode}  r_cut_b={r_cut_b:.3f}")
print(f"BEFORE: n={len(before_pos)}  AA={int((before_typ==TYPE_AA).sum())}  CG={int((before_typ==TYPE_CG).sum())}  -> {before_path}")
print(f"AFTER:  n={len(after_pos)}  AA={int((after_typ==TYPE_AA).sum())}  CG={int((after_typ==TYPE_CG).sum())}  phantom/surv={n_ph}  -> {after_path}")

center_i=31  mode=unpack (situation A)  r_cut_b=6.719
BEFORE: n=55  AA=53  CG=2  -> out/nbr_boundary_31_before.xyz
AFTER:  n=55  AA=53  CG=2  phantom/surv=0  -> out/nbr_boundary_31_after.xyz
